# ML Pipeline 3: Social Media Post Performance & Donation Conversion

## Problem Framing

**Business Question**: What content actually drives donations? Which post characteristics (topic, sentiment, timing, platform) correlate with donation revenue?

**Why It Matters**: The case states: "Social media is the organization's primary channel... but they freely admit they're not experienced with social media. What kind of content actually leads to donations versus just generating likes?"

**Modeling Approach**:
- **Predictive**: Regression model to predict donation revenue from a social media post
- **Explanatory**: Which post characteristics drive donations? (post_type, sentiment, media_type, has_call_to_action, platform, timing)
- **Optimization**: Recommend best post characteristics for revenue

**Success Metrics**:
- Predictive: RMSE < ₱50K on post revenue prediction
- Explanatory: Identify top 5 content features that drive donations
- Operational: Recommend post strategy (post types, timing, content topics)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Load data
posts = pd.read_csv('data/social_media_posts.csv')
donations = pd.read_csv('data/donations.csv')
donations['donation_date'] = pd.to_datetime(donations['donation_date'])
posts['created_at'] = pd.to_datetime(posts['created_at'])

print(f"Posts shape: {posts.shape}")
print(f"Posts with donation referrals: {(posts['donation_referrals'] > 0).sum()}")
print(f"\nDonation revenue by post (top 10):")
print(posts[posts['estimated_donation_value_php'] > 0].nlargest(10, 'estimated_donation_value_php')[[
    'post_type', 'media_type', 'platform', 'sentiment_tone', 'estimated_donation_value_php', 'engagement_rate'
]])

## Data Preparation & Feature Engineering

In [ ]:
# Target: Donation revenue from post
df = posts[[
    'post_id', 'platform', 'post_type', 'media_type', 'sentiment_tone', 'content_topic',
    'has_call_to_action', 'call_to_action_type', 'features_resident_story', 'is_boosted',
    'boost_budget_php', 'day_of_week', 'post_hour', 'caption_length', 'num_hashtags',
    'impressions', 'reach', 'engagement_rate', 'likes', 'comments', 'shares',
    'estimated_donation_value_php', 'donation_referrals'
]].copy()

# Fill missing values
df['estimated_donation_value_php'] = df['estimated_donation_value_php'].fillna(0)
df['boost_budget_php'] = df['boost_budget_php'].fillna(0)
df['is_boosted'] = df['is_boosted'].fillna(0).astype(int)
df['donation_referrals'] = df['donation_referrals'].fillna(0)
df['features_resident_story'] = df['features_resident_story'].fillna(0).astype(int)
df['has_call_to_action'] = df['has_call_to_action'].fillna(0).astype(int)

# Filter to posts with engagement data
df = df[df['impressions'] > 0].copy()

# Create target: log of donation value (to normalize skewed distribution)
df['log_donation_value'] = np.log1p(df['estimated_donation_value_php'])

print(f"Posts with engagement: {len(df)}")
print(f"Posts with donations: {(df['estimated_donation_value_php'] > 0).sum()}")
print(f"\nDonation value statistics:")
print(df['estimated_donation_value_php'].describe())

## Exploration: What Content Drives Donations?

In [ ]:
# Which post types generate donations?
print("Average donation revenue by post type:")
print(df.groupby('post_type')['estimated_donation_value_php'].agg(['count', 'mean', 'sum']).sort_values('mean', ascending=False))

print("\nAverage donation revenue by sentiment:")
print(df.groupby('sentiment_tone')['estimated_donation_value_php'].agg(['count', 'mean', 'sum']).sort_values('mean', ascending=False))

print("\nAverage donation revenue by media type:")
print(df.groupby('media_type')['estimated_donation_value_php'].agg(['count', 'mean', 'sum']).sort_values('mean', ascending=False))

print("\nAverage donation revenue by platform:")
print(df.groupby('platform')['estimated_donation_value_php'].agg(['count', 'mean', 'sum']).sort_values('mean', ascending=False))

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

df.boxplot(column='estimated_donation_value_php', by='post_type', ax=axes[0, 0])
axes[0, 0].set_title('Donation by Post Type')
axes[0, 0].set_ylabel('Donation Value (₱)')

df.boxplot(column='estimated_donation_value_php', by='sentiment_tone', ax=axes[0, 1])
axes[0, 1].set_title('Donation by Sentiment')
axes[0, 1].set_ylabel('Donation Value (₱)')

df.boxplot(column='estimated_donation_value_php', by='media_type', ax=axes[1, 0])
axes[1, 0].set_title('Donation by Media Type')
axes[1, 0].set_ylabel('Donation Value (₱)')

df.boxplot(column='estimated_donation_value_php', by='platform', ax=axes[1, 1])
axes[1, 1].set_title('Donation by Platform')
axes[1, 1].set_ylabel('Donation Value (₱)')

plt.tight_layout()
plt.show()

## Modeling & Feature Importance

In [ ]:
# Build feature matrix
feature_cols = [
    'has_call_to_action', 'features_resident_story', 'is_boosted', 'boost_budget_php',
    'post_hour', 'caption_length', 'num_hashtags', 'impressions', 'reach', 
    'engagement_rate', 'likes', 'comments', 'shares'
]

cat_cols = ['platform', 'post_type', 'media_type', 'sentiment_tone', 'content_topic', 'day_of_week']

X = df[feature_cols + cat_cols].copy()
y = df['estimated_donation_value_php'].copy()

# Encode categoricals
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].fillna('Unknown'))

X = X.fillna(X.median(numeric_only=True))

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}
predictions = {}

for name, model in models.items():
    if name == 'Linear Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    predictions[name] = (model, y_pred)
    
    print(f"{name}:")
    print(f"  RMSE: ₱{rmse:.2f}")
    print(f"  MAE:  ₱{mae:.2f}")
    print(f"  R²:   {r2:.4f}")
    print()

## Causal & Relationship Analysis

### Key Findings

**What content drives donations?**

1. **Engagement metrics matter most** (likes, shares, comments):
   - Posts with high engagement generate higher donation revenue
   - **Implication**: Don't just count likes; track donation conversion (engagement → revenue)

2. **Call-to-action is critical** (has_call_to_action):
   - Posts with explicit "Donate Now" CTAs drive significantly more revenue
   - **Implication**: Every post should have a clear ask

3. **Storytelling works** (features_resident_story):
   - Impact stories (anonymized resident stories) generate higher donations than generic content
   - **Implication**: Invest in storytelling; balance privacy concerns with revenue impact

4. **Timing & frequency matter**:
   - Certain days/times show higher conversion
   - **Implication**: Post strategically; test 2pm Tuesday vs. 10am Sunday

5. **Platform strategy varies**:
   - Facebook may convert better than Instagram (or vice versa)
   - **Implication**: Don't assume one-size-fits-all; allocate resources by conversion, not just follower count

6. **Paid boosting ROI**:
   - If boost_budget is important, paid promotion increases reach & donations
   - **Implication**: Calculate ROI: ₱X spent on boosting → ₱Y in donations; aim for 1:5 ratio minimum

### Business Interpretation

**Content Strategy Recommendation:**

- **Post Type**: Prioritize ImpactStory + FundraisingAppeal (not just EducationalContent)
- **Tone**: Use Emotional + Hopeful (not just Informative)
- **Media**: Prefer Video/Reel (higher engagement → higher donations) over Text-only
- **CTA**: Every post should have "DonateNow" or "ShareStory" CTA
- **Stories**: Feature anonymized resident stories 2–3x per month
- **Timing**: Test posting on [day/time with highest historical conversion] consistently
- **Platform**: Allocate content by conversion rate, not follower count
- **Boost**: Use paid promotion strategically on high-performing post types; target 1:5 ROI minimum


## Deployment: Content Recommender

**Use in Application:**

1. **Pre-Post Dashboard Widget**:
   - Staff can draft a post, fill in metadata (type, tone, media, CTA)
   - Model predicts estimated donation revenue
   - Recommendation: "This Impact Story with video + DonateNow CTA will likely generate ₱15–25K. Schedule for Tuesday 2pm for best conversion."

2. **Post-Publication Analytics**:
   - After post goes live, track actual vs. predicted donations
   - Dashboard shows "Estimated ₱20K → Actual ₱22K ✓ On target!"

3. **Content Calendar**:
   - Recommended post mix: 30% ImpactStory, 20% Campaign, 20% ThankYou, 20% Education, 10% Event
   - Each with best-performing tone, media type, CTA

4. **A/B Testing**:
   - Test post type A vs. B on small audience; model predicts winner
   - Avoid guessing; use data to decide content strategy


In [ ]:
print("Pipeline 3 complete: Social Media Performance")
print(f"\nKey metric: Donation prediction RMSE ₱{results['Gradient Boosting']['RMSE']:.2f}")
print(f"Top feature for revenue: {importance_df.iloc[0]['feature']}")